In [5]:
import pandas as pd

races=pd.read_csv('races.csv')
laps=pd.read_csv('lap_times.csv')
results=pd.read_csv('results.csv')
circuits=pd.read_csv('circuits.csv')

races=races[races['year']>=2018] # Filter

valid_race_ids = races['raceId']
# Filter laps and results to include only valid raceIds
laps = laps[laps['raceId'].isin(valid_race_ids)]
results = results[results['raceId'].isin(valid_race_ids)]
# Save the cleaned datasets to new CSV files
races.to_csv("races_cleaned.csv", index=False)
laps.to_csv("laps_clean.csv", index=False)
results.to_csv("results_clean.csv", index=False)

CHECKING BASIC COUNTS

In [4]:
print("Races:", len(races))
print("Laps:", len(laps))
print("Results:", len(results))

Races: 149
Laps: 162448
Results: 2979


CHECK YEAR FILLTER ACTUALLY WORKED

In [5]:
print(races['year'].min(), races['year'].max())

2018 2024


VERIFY RACEID CONSISTENCY

In [6]:
print(laps['raceId'].isin(races['raceId']).all())
print(results['raceId'].isin(races['raceId']).all())

True
True


VERIFY BY ONE RACEID

In [7]:
race = races.iloc[5]

print(race[['raceId','year','name','circuitId']])

raceId                     994
year                      2018
name         Monaco Grand Prix
circuitId                    6
Name: 981, dtype: object


In [8]:
race_laps = laps[laps['raceId'] == race['raceId']]
print(race_laps.head())
print(race_laps.tail())

        raceId  driverId  lap  position      time  milliseconds
431539     994       817    1         1  1:22.199         82199
431540     994       817    2         1  1:18.135         78135
431541     994       817    3         1  1:17.810         77810
431542     994       817    4         1  1:17.796         77796
431543     994       817    5         1  1:17.549         77549
        raceId  driverId  lap  position      time  milliseconds
433047     994       154   73        15  1:19.000         79000
433048     994       154   74        15  1:14.822         74822
433049     994       154   75        15  1:18.895         78895
433050     994       154   76        15  1:19.133         79133
433051     994       154   77        15  1:21.209         81209


CHECK LAP COVERAGE

In [9]:
print(race_laps['lap'].min(), race_laps['lap'].max())

1 78


CHECK POSITIONS ARE VALID

In [10]:
print(race_laps['position'].unique())

[ 1  2  3  6  5  4  7  8 10  9 15 13 12 11 14 20 19 18 16 17]


PICK A RACE DYNAMICALLY

In [6]:
race = races.sample(1).iloc[0]


race_id = race['raceId']
year = race['year']
circuit_id = race['circuitId']

print(f"Selected: {race['name']} ({year}) raceId:{race_id}")

Selected: Italian Grand Prix (2024) raceId:1136


GET CIRCUIT INFO

In [7]:
circuit = circuits[circuits['circuitId'] == circuit_id].iloc[0]

circuit_name_csv = circuit['name']
print("Circuit:", circuit_name_csv)

Circuit: Autodromo Nazionale di Monza


AUTO MAP TO FastF1 TRACK

In [8]:
import fastf1

def map_to_fastf1(name):
    name = name.lower()

    if "albert park" in name:
        return "Australia"
    elif "monaco" in name:
        return "Monaco"
    elif "silverstone" in name:
        return "Silverstone"
    elif "spa" in name:
        return "Spa"
    elif "monza" in name:
        return "Monza"
    elif "suzuka" in name:
        return "Suzuka"
    elif "istanbul" in name:
        return "Turkey"
    elif "bahrain" in name:
        return "Bahrain"
    elif "hungaroring" in name:
        return "Hungary"
    else:
        return None

track_name=map_to_fastf1(circuit_name_csv)
if track_name is None:
    print(f"Track not supported: {circuit_name_csv}")

LOAD FastF1 AUTOMATICALLY

In [32]:
session = fastf1.get_session(year, track_name, 'R')
session.load()

core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

In [33]:
race_laps = laps[laps['raceId'] == race_id]
drivers = race_laps['driverId'].unique()[:5]

In [34]:
fastf1.get_session(year, "Yas Marina Circuit", "R")

events      WARNING 	Correcting user input 'Yas Marina Circuit' to 'Mexican Grand Prix'


2019 Season Round 18: Mexican Grand Prix - Race

AUTO DETECT FROM FastF1

In [9]:
schedule = fastf1.get_event_schedule(2018) # Get the schedule for the 2023 season
print(schedule[['EventName', 'Location']])

req         WARNING 	DEFAULT CACHE ENABLED! (415.05 MB) /home/abhi/.cache/fastf1


                   EventName           Location
0      Australian Grand Prix          Melbourne
1         Bahrain Grand Prix             Sakhir
2         Chinese Grand Prix           Shanghai
3      Azerbaijan Grand Prix               Baku
4         Spanish Grand Prix          Barcelona
5          Monaco Grand Prix        Monte Carlo
6        Canadian Grand Prix           Montréal
7          French Grand Prix       Le Castellet
8        Austrian Grand Prix          Spielberg
9         British Grand Prix        Silverstone
10         German Grand Prix         Hockenheim
11      Hungarian Grand Prix           Budapest
12        Belgian Grand Prix  Spa-Francorchamps
13        Italian Grand Prix              Monza
14      Singapore Grand Prix          Singapore
15        Russian Grand Prix              Sochi
16       Japanese Grand Prix             Suzuka
17  United States Grand Prix             Austin
18        Mexican Grand Prix        Mexico City
19      Brazilian Grand Prix          Sã

In [10]:
race = races.sample(1).iloc[0]

race_id = race['raceId']
circuit_id = race['circuitId']

# get correct circuit for THIS race
circuit = circuits[circuits['circuitId'] == circuit_id].iloc[0]

race_name = race['name']
location = circuit['location']

print(f"race_name: {race_name}, location: {location}, year: {race['year']}")

race_name: Belgian Grand Prix, location: Spa, year: 2019


In [11]:
def find_event(year, race_name, location): # Find the event name in FastF1 schedule that matches the given
    schedule = fastf1.get_event_schedule(year)

    race_name = race_name.lower()
    location = location.lower()

    for _, event in schedule.iterrows(): # Iterate through the schedule to find a matching event
        event_name = str(event['EventName']).lower() # Convert to string and lowercase for comparison
        event_loc = str(event['Location']).lower()

        # match by race name OR location
        if race_name in event_name or location in event_loc:
            return event['EventName']

    return None

In [12]:
event_name = find_event(year, race['name'], circuit['location']) # Find the matching event name in FastF1 schedule

if event_name is None:
    raise ValueError("No matching FastF1 event found")

print("Matched Event:", event_name)

Matched Event: Belgian Grand Prix


In [13]:
session = fastf1.get_session(year, event_name, 'R') # Use the matched event name to load the session
session.load()

core           INFO 	Loading data for Belgian Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

ADD CONFIDENCE SCORE

In [14]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

In [15]:
def find_event(year, race_name, location, threshold=0.6):
    schedule = fastf1.get_event_schedule(year)

    race_name = race_name.lower()
    location = location.lower()

    best_match = None
    best_score = 0

    for _, event in schedule.iterrows():
        event_name = str(event['EventName']).lower()
        event_loc = str(event['Location']).lower()

        score_name = similarity(race_name, event_name)
        score_loc = similarity(location, event_loc)

        score = max(score_name, score_loc)

        if score > best_score:
            best_score = score
            best_match = event['EventName']

    if best_score >= threshold:
        return best_match, best_score
    else:
        return None, best_score

In [16]:
# Find the matching event name in FastF1 schedule and also get the similarity score
event_name, score = find_event(year, race['name'], circuit['location']) 

print("CSV Race:", race['name'])
print("Location:", circuit['location'])
print("Matched:", event_name)
print("Confidence:", score)

CSV Race: Belgian Grand Prix
Location: Spa
Matched: Belgian Grand Prix
Confidence: 1.0


CROSS CHECK WITH CIRCUIT

In [17]:
schedule = fastf1.get_event_schedule(year)
# Get the row for the matched event
event_row = schedule[schedule['EventName'] == event_name].iloc[0] 

print("FastF1 Location:", event_row['Location'])

FastF1 Location: Spa-Francorchamps


In [18]:
for i in range(10):
    race = races.sample(1).iloc[0]
    # Get the circuit details for the selected race
    circuit = circuits[circuits['circuitId'] == race['circuitId']].iloc[0] 
    
    # Find the matching event name in FastF1 schedule and also get the similarity score
    event_name, score = find_event(race['year'], race['name'], circuit['location']) 

    print(race['name'], "→", event_name, "|", score)

São Paulo Grand Prix → São Paulo Grand Prix | 1.0
Emilia Romagna Grand Prix → Emilia Romagna Grand Prix | 1.0
Qatar Grand Prix → Qatar Grand Prix | 1.0
Belgian Grand Prix → Belgian Grand Prix | 1.0
Spanish Grand Prix → Spanish Grand Prix | 1.0
São Paulo Grand Prix → São Paulo Grand Prix | 1.0
Russian Grand Prix → Russian Grand Prix | 1.0
Azerbaijan Grand Prix → Azerbaijan Grand Prix | 1.0
French Grand Prix → French Grand Prix | 1.0
Belgian Grand Prix → Belgian Grand Prix | 1.0
